# GPU benchmarks: one group at a time, no Google Drive
**Run all runs only the ONE selected group.** Each group runs 150 requests × 5 strategies × 3 repetitions (2,250 request executions), in float32, with four generated tokens per request. This is one sixth of the previous full suite, not a promise of a fixed runtime.

1. Select a T4 runtime.
2. Choose one group in the dropdown below (start with 1).
3. Run all. Live progress appears during the benchmark.
4. Allow the browser download and confirm the group ZIP exists in your laptop's Downloads before closing the laptop.
5. In a later session, select the next unfinished group and repeat. In the same session, change the selector, run that selector cell, then rerun the benchmark cell.

No Drive is mounted. Model weights stay in temporary Colab storage; only experiment results are zipped. Closing the laptop can still lose the current unfinished group, but earlier downloaded ZIPs remain safe. Keep all six ZIPs and send them for analysis. Do not change settings or tune the predictor based on these test results.


In [ ]:
#@title Choose ONE group to run
GROUP = "1: hot / 8 MiB" #@param ["1: hot / 8 MiB", "2: hot / 32 MiB", "3: shift / 8 MiB", "4: shift / 32 MiB", "5: scan / 8 MiB", "6: scan / 32 MiB"]
groups = {1: ("hot", 8), 2: ("hot", 32), 3: ("shift", 8),
          4: ("shift", 32), 5: ("scan", 8), 6: ("scan", 32)}
group_number = int(GROUP.split(":", 1)[0])
workload, budget = groups[group_number]
print(f"Selected group {group_number}/6: {workload}, {budget} MiB. Only this group will run.")


In [ ]:
import os, pathlib, subprocess, sys
os.environ["USE_TF"] = "0"
repo = pathlib.Path("/content/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/swathiblrs/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference.git", str(repo)], check=True)
os.chdir(repo)
# Keep Colab's Gradio/Diffusers dependencies intact; reuse its installed CUDA torch.
# virtualenv supplies pip itself; it does not depend on system ensurepip.
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv>=20.26,<21"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages",
                "--no-download", ".colab-venv"], check=True)
python = str(repo / ".colab-venv/bin/python")
subprocess.run([python, "-m", "pip", "install", "-e", ".[inference]"], check=True)


In [ ]:
subprocess.run([python, "-c", "import torch; assert torch.cuda.is_available(), 'Select an NVIDIA GPU runtime'; print(torch.cuda.get_device_name())"], check=True)

subprocess.run([python, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Run selected group and download
This starts only the selected group. Keep this cell running until the download starts. Output equality remains mandatory; failed runs are downloaded for diagnosis, not counted as successful groups. A new uniquely named output folder avoids mixing or overwriting earlier runs. Each group includes a no-cache baseline measured in the same session. One held-out seed (99) is used; repetitions are timing trials, not independent workload samples.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from google.colab import files
import json
import statistics

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
run_dir = Path("/content") / f"kv-cache-group{group_number}-{workload}-{budget}MiB-{stamp}"
archive = str(run_dir) + ".zip"
command = [python, "-u", "-m", "kv_cache.suite", "--output", str(run_dir),
           "--requests", "150", "--repeats", "3", "--budgets", str(budget),
           "--workloads", workload, "--device", "cuda", "--seed", "99"]
print("Results:", run_dir, flush=True)
print("One group only. Wait for the download before leaving.", flush=True)
try:
    # Stream subprocess output into the cell so progress is visible in Colab.
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError("Group failed; download diagnostics and send them for investigation.")
    manifest = json.loads((run_dir / "manifest.json").read_text())
    report = json.loads((run_dir / f"{workload}-{budget:g}MiB.json").read_text())
    assert manifest["status"] == "complete" and report["complete"] and report["greedy_outputs_match"]
    print("\nCORRECTNESS PASSED. Summary: median of per-run p95 values (not a pooled p95).")
    for policy in ("none", "fifo", "lru", "lfu", "learned"):
        rows = [r for r in report["runs"] if r["policy"] == policy]
        p95s = [r["ttft_p95_ms"] for r in rows]
        print(f"{policy:8s} hits={rows[0]['hits']}/{rows[0]['requests']} "
              f"median run-p95={statistics.median(p95s):.2f} ms "
              f"range=[{min(p95s):.2f}, {max(p95s):.2f}] ms")
finally:
    if Path(archive).exists():
        print("Downloading", Path(archive).name)
        files.download(archive)
    else:
        print("No archive available yet. Keep the runtime open and inspect the error above.")


## If your browser blocks the download
Run the cell below to retry downloading the same ZIP. **Do not rerun the benchmark just to download it.** Alternatively, find the ZIP under `/content` in Colab's Files panel and select Download. Confirm the file is saved locally before ending the session.

Completed groups do not need to be uploaded back to Colab. Keep them on your laptop; choose another group next time. If a group fails correctness, stop and send its ZIP before proceeding. Results from different GPU types or software versions must be identified during analysis rather than treated as identical environments.


In [ ]:
#@title Optional: retry a blocked download (leave unchecked normally)
RETRY_DOWNLOAD = False #@param {type:"boolean"}
if RETRY_DOWNLOAD:
    if "archive" in globals() and Path(archive).exists():
        files.download(archive)
    else:
        print("No archive found in this session. Check /content in the Files panel.")
